# BITS Pilani WILP – Machine Learning Assignment 2
## Dry Bean Classification

Run every code cell from top to bottom in BITS Virtual Lab.

This notebook creates `dry_bean_dataset.csv`, `test_data.csv`, `metrics.csv`, `feature_columns.json`, and five saved models inside `model/`. These files are used by `app.py`.

## 1. Install required packages

In [ ]:
%pip install -q ucimlrepo scikit-learn pandas numpy matplotlib joblib streamlit

## 2. Import libraries and create the model folder

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

PROJECT_DIR = Path.cwd()
MODEL_DIR = PROJECT_DIR / "model"
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
print("Project folder:", PROJECT_DIR)
print("Model folder:", MODEL_DIR)

## 3. Load and inspect the UCI Dry Bean dataset

In [ ]:
dry_bean = fetch_ucirepo(id=602)
X = dry_bean.data.features.copy()
y = dry_bean.data.targets.iloc[:, 0].copy()
y.name = "Class"

print("Number of instances:", X.shape[0])
print("Number of features:", X.shape[1])
print("Feature names:", list(X.columns))
display(y.value_counts().rename_axis("Class").to_frame("Count"))

full_dataset = X.copy()
full_dataset["Class"] = y
full_dataset.to_csv(PROJECT_DIR / "dry_bean_dataset.csv", index=False)
display(full_dataset.head())

## 4. Split into training and testing sets (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Number of features:", X_train.shape[1])

## 5. Define the five classification models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(max_depth=12, min_samples_split=8, random_state=RANDOM_STATE))
    ]),
    "K-Nearest Neighbors": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Naive Bayes": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", GaussianNB())
    ]),
    "Support Vector Machine": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", C=3.0, probability=True, decision_function_shape="ovr", random_state=RANDOM_STATE))
    ])
}
print("Five models defined:")
for name in models:
    print("-", name)

## 6. Train models, calculate all six metrics and save models

In [ ]:
results = []
for model_name, model in models.items():
    print("Training:", model_name)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)

    auc = roc_auc_score(
        y_test, probabilities, multi_class="ovr",
        average="weighted", labels=model.classes_
    )
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": auc,
        "Precision": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "F1": f1_score(y_test, predictions, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions)
    })

    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, MODEL_DIR / filename)
    print("Saved model/" + filename)

metrics_df = pd.DataFrame(results).sort_values("F1", ascending=False)
metrics_df.to_csv(PROJECT_DIR / "metrics.csv", index=False)
display(metrics_df.round(4))

## 7. Save test data for the Streamlit app

In [ ]:
test_data = X_test.copy()
test_data["Class"] = y_test
test_data.to_csv(PROJECT_DIR / "test_data.csv", index=False)

with open(PROJECT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(list(X.columns), f, indent=2)

print("Created test_data.csv")
print("Created metrics.csv")
print("Created feature_columns.json")
display(test_data.head())

## 8. Compare models and identify the leading result

In [ ]:
display(metrics_df.sort_values("F1", ascending=False).reset_index(drop=True))
winner = metrics_df.sort_values(["F1", "Accuracy", "MCC"], ascending=False).iloc[0]
print("Leading model by F1 (accuracy and MCC used as tie-breakers):", winner["Model"])
print("Use the actual metrics above to write your assignment observations.")

## 9. Verify all generated files

In [ ]:
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file() and ".ipynb_checkpoints" not in str(path):
        print(path.relative_to(PROJECT_DIR))